# MGMT298D: Science and Strategy of AI
### Week 2 - Tree-based Predictions

### *Application: Vehicle Pricing*

**Student Name:**

---

## Import Libraries and Load Data


In [ ]:
import pandas as pd
import numpy as np
from sklearn import linear_model, metrics
from sklearn.model_selection import train_test_split  # Added for splitting
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/range_rover.csv")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Data loaded and split successfully!")
print(f"Total rows: {len(df)}")
print(f"Training set: {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test set: {test_df.shape[0]} rows, {test_df.shape[1]} columns")

Data loaded successfully!
Training set: 263 rows, 9 columns
Test set: 112 rows, 9 columns


### Data Preparation

In [ ]:
# For tree-based models, we'll use label encoding instead of dummy variables
def prepare_tree_data(df_train, df_test):
    # Create copies to avoid modifying original data
    train_encoded = df_train.copy()
    test_encoded = df_test.copy()

    # Initialize label encoders
    encoders = {}
    categorical_cols = ['trim', 'state', 'color']

    # Fit encoders on combined data to ensure consistency
    for col in categorical_cols:
        encoders[col] = LabelEncoder()
        combined_values = pd.concat([df_train[col], df_test[col]]).unique()
        encoders[col].fit(combined_values)

        # Transform both datasets
        train_encoded[f'{col}_encoded'] = encoders[col].transform(df_train[col])
        test_encoded[f'{col}_encoded'] = encoders[col].transform(df_test[col])

    # Select features for modeling
    feature_cols = ['year', 'mileage'] + [f'{col}_encoded' for col in categorical_cols]

    X_train = train_encoded[feature_cols]
    X_test = test_encoded[feature_cols]
    y_train = df_train['sellingprice']
    y_test = df_test['sellingprice']

    return X_train, X_test, y_train, y_test, encoders

X_train, X_test, y_train, y_test, encoders = prepare_tree_data(train_df, test_df)

print("Data prepared for tree-based modeling:")
print(f"Features: {list(X_train.columns)}")
print(f"Training examples: {len(X_train)}")
print(f"Test examples: {len(X_test)}")

Data prepared for tree-based modeling:
Features: ['year', 'mileage', 'trim_encoded', 'state_encoded', 'color_encoded']
Training examples: 263
Test examples: 112


## Model 1: Regression Tree

In [ ]:
# Train a Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42, max_depth=10, min_samples_split=20, min_samples_leaf=10)
dt_model.fit(X_train, y_train)

# Make predictions
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

# Calculate metrics
dt_train_mae = metrics.mean_absolute_error(y_train, dt_train_pred)
dt_test_mae = metrics.mean_absolute_error(y_test, dt_test_pred)

print("=== DECISION TREE RESULTS ===")
print(f"Training MAE: ${dt_train_mae:,.2f}")
print(f"Test MAE: ${dt_test_mae:,.2f}")
print(f"Overfitting gap (MAE): ${dt_test_mae - dt_train_mae:,.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
for idx, row in feature_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

=== DECISION TREE RESULTS ===
Training MAE: $3,013.44
Test MAE: $4,186.65
Overfitting gap (MAE): $1,173.21

Feature Importance:
  year: 0.9267
  mileage: 0.0427
  trim_encoded: 0.0304
  color_encoded: 0.0002
  state_encoded: 0.0001


## Model 2: Random Forest


In [ ]:
# Train a Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=15,
                                   min_samples_split=10, min_samples_leaf=5, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# Calculate metrics
rf_train_mae = metrics.mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = metrics.mean_absolute_error(y_test, rf_test_pred)

print("=== RANDOM FOREST RESULTS ===")
print(f"Training MAE: ${rf_train_mae:,.2f}")
print(f"Test MAE: ${rf_test_mae:,.2f}")
print(f"Overfitting gap (MAE): ${rf_test_mae - rf_train_mae:,.2f}")

# Feature importance
rf_feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
for idx, row in rf_feature_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

=== RANDOM FOREST RESULTS ===
Training MAE: $2,266.20
Test MAE: $3,500.74
Overfitting gap (MAE): $1,234.53

Feature Importance:
  year: 0.9334
  mileage: 0.0445
  trim_encoded: 0.0204
  state_encoded: 0.0009
  color_encoded: 0.0008


## Model 3: XGBoost


In [ ]:
# Train an XGBoost Regressor
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, max_depth=6,
                             learning_rate=0.1, subsample=0.8, colsample_bytree=0.8)
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Calculate metrics
xgb_train_mae = metrics.mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = metrics.mean_absolute_error(y_test, xgb_test_pred)

print("=== XGBOOST RESULTS ===")
print(f"Training MAE: ${xgb_train_mae:,.2f}")
print(f"Test MAE: ${xgb_test_mae:,.2f}")
print(f"Overfitting gap (MAE): ${xgb_test_mae - xgb_train_mae:,.2f}")

# Feature importance
xgb_feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
for idx, row in xgb_feature_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

=== XGBOOST RESULTS ===
Training MAE: $523.99
Test MAE: $3,405.47
Overfitting gap (MAE): $2,881.48

Feature Importance:
  year: 0.9437
  mileage: 0.0286
  trim_encoded: 0.0220
  state_encoded: 0.0030
  color_encoded: 0.0028


## Model Comparison and Analysis

Let's compare the baseline performance of all three models to identify the best starting point.

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Train_MAE': [dt_train_mae, rf_train_mae, xgb_train_mae],
    'Test_MAE': [dt_test_mae, rf_test_mae, xgb_test_mae],
    'Overfitting_Gap_MAE': [dt_test_mae - dt_train_mae, rf_test_mae - rf_train_mae, xgb_test_mae - xgb_train_mae]
})

print("=== BASELINE MODEL COMPARISON ===")
print(results_df.round(2))

# Identify best model based on test MAE
best_model_idx = results_df['Test_MAE'].idxmin()
best_baseline_model_name = results_df.loc[best_model_idx, 'Model']
best_baseline_test_mae = results_df.loc[best_model_idx, 'Test_MAE']

print(f"\n=== BEST BASELINE MODEL: {best_baseline_model_name} ===")
print(f"Test MAE: ${best_baseline_test_mae:,.2f}")

=== BASELINE MODEL COMPARISON ===
           Model  Train_MAE  Test_MAE  Overfitting_Gap_MAE
0  Decision Tree    3013.44   4186.65              1173.21
1  Random Forest    2266.20   3500.74              1234.53
2        XGBoost     523.99   3405.47              2881.48

=== BEST BASELINE MODEL: XGBoost ===
Test MAE: $3,405.47


## Questions for Analysis

**1. Compare Performance**: How do the tree-based models compare to each other and to linear regression from Lab 1 in terms of Mean Absolute Error (MAE)? What are the trade-offs?

**2. Feature Importance**: Which features are most important for pricing predictions? How does this compare to your business intuition?

**3. Overfitting Analysis**: Which model shows the best balance between training and test performance (i.e., the smallest gap in MAE)? What does this tell you about model generalization?

**4. Business Applications**: Based on your results, what specific recommendations would you make to a used car dealership for:
  - Inventory pricing strategy
  - Acquisition decisions
  - Market positioning

**5. Model Selection**: If you had to choose one model for production use, which would you choose and why? Consider accuracy (MAE), interpretability, and computational requirements.

---

## Conclusion

In this lab, you've successfully implemented and compared three tree-based machine learning models for automotive pricing. You've learned how these non-linear methods can capture complex relationships in data and provide valuable business insights through feature importance analysis. The skills you've developed here form the foundation for more advanced machine learning applications in business strategy and operations.